# APD generation shift

One coauthor running this notebook once per day on **Colab T4 (90 min/day)** or **Kaggle GPU (12 h shift)** produces ~50–675 images per shift, idempotently. Three coauthors × two platforms × ~12 days clears the 12 000-image main grid plus the 3 200-image robustness grid.

**How it works.** Each shift:
1. Loads the canonical `main_cells()` grid from the repo (25 occ × 4 lang × 4 models × 30 imgs = 12 000 cells).
2. Reads `images/main/metadata.parquet` to see which cells are already done.
3. Picks the next `BUDGET` pending cells.
4. Generates them (Pollinations.ai for FLUX cells; local `diffusers` for SD cells).
5. Appends to `metadata.parquet` and saves PNGs.
6. Reports cumulative progress.

**Before running.** Set the three variables in the *Configuration* cell below:
* `REPO_URL` — your fork of the apd-audit repo (public GitHub URL).
* `HF_TOKEN` — optional, only needed if HF Inference comes back.
* `BUDGET` — cells to generate this shift. 50 is safe for Pollinations on a Colab T4; bump to 200+ on a Kaggle GPU running local SD.

In [ ]:
# === Configuration — edit these three lines ===
REPO_URL  = "https://github.com/hlaverde/apd-audit.git"
HF_TOKEN  = ""   # leave empty for Pollinations
BUDGET    = 50   # cells per shift

In [ ]:
# Detect environment and configure persistent storage.
import os, sys, subprocess, pathlib

IS_COLAB  = "google.colab" in sys.modules
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print(f"Colab: {IS_COLAB} | Kaggle: {IS_KAGGLE}")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive/apd-audit")
elif IS_KAGGLE:
    BASE = pathlib.Path("/kaggle/working/apd-audit")
else:
    BASE = pathlib.Path.cwd() / "apd-audit"

BASE.parent.mkdir(parents=True, exist_ok=True)
print("Project root:", BASE)

In [ ]:
# Clone or pull the repository.
if BASE.exists():
    subprocess.run(["git", "-C", str(BASE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(BASE)], check=True)

sys.path.insert(0, str(BASE / "src"))
os.environ["PYTHONPATH"] = str(BASE / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("Repo cloned/updated.")

In [ ]:
# Install the runtime dependencies. We install directly with pip rather
# than uv to avoid the Colab/Kaggle overhead of bootstrapping uv.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "pandas>=2.2", "pyarrow>=15", "numpy>=1.26,<3", "scipy>=1.13",
        "pydantic>=2.7", "pydantic-settings>=2.3", "python-dotenv>=1",
        "requests>=2.32", "huggingface-hub>=0.25",
        "pillow>=10", "opencv-python>=4.9",
        "skin-tone-classifier>=1.2.3",
        "matplotlib>=3.8",
    ],
    check=True,
)
print("Deps installed.")

In [ ]:
# Optional: install the heavy ML extras when this shift will generate SD
# cells (not needed if you're only generating Pollinations FLUX cells).
WANT_LOCAL_DIFFUSERS = True
if WANT_LOCAL_DIFFUSERS:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "torch>=2.2", "diffusers>=0.30", "transformers>=4.42",
            "accelerate>=0.33", "safetensors>=0.4",
        ],
        check=True,
    )
    print("ML extras installed.")

In [ ]:
# Load the canonical grid and figure out what's left to do.
import pandas as pd
from apd.prompts.grid import main_cells, expected_main_grid_size
from apd.generate.orchestrator import image_path, generate_poc

IMAGES_DIR = BASE / "images" / "main"
META_FILE  = IMAGES_DIR / "metadata.parquet"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

done_image_ids: set[str] = set()
if META_FILE.exists():
    done = pd.read_parquet(META_FILE)
    done_image_ids = set(done["image_id"])

all_cells = list(main_cells())
pending = []
for c in all_cells:
    img_id = f"{c.model.replace('/', '_')}__{c.occupation.replace(' ', '_')}__{c.seed}"
    if img_id not in done_image_ids:
        pending.append(c)
    if len(pending) >= BUDGET:
        break

print(f"Main grid total      : {expected_main_grid_size():>6} cells")
print(f"Already done         : {len(done_image_ids):>6}")
print(f"This shift will do   : {len(pending):>6} (BUDGET = {BUDGET})")

In [ ]:
# Generate. The orchestrator picks the backend for each cell from its
# model identifier:
#   pollinations/flux           → PollinationsBackend (no token, free)
#   runwayml/...                → LocalBackend via diffusers (ml extras)
#   stabilityai/...             → LocalBackend via diffusers (ml extras)
import time

start = time.time()
shard = generate_poc(pending, out_dir=IMAGES_DIR)
elapsed = time.time() - start
print(f"Generated {len(shard)} images in {elapsed:.0f}s ({elapsed/max(len(shard),1):.1f}s/img).")

In [ ]:
# Merge this shard into the running metadata.parquet and write it back.
if META_FILE.exists():
    existing = pd.read_parquet(META_FILE)
    combined = (
        pd.concat([existing, shard], ignore_index=True)
        .drop_duplicates(subset="image_id", keep="last")
    )
else:
    combined = shard
combined.to_parquet(META_FILE, index=False)

remaining = expected_main_grid_size() - len(combined)
pct = 100.0 * len(combined) / max(expected_main_grid_size(), 1)
print(f"Metadata file has {len(combined)} images ({pct:.1f}% of grid).")
print(f"Remaining cells     : {remaining}")

## End of shift

After this notebook finishes, the PNG files and the updated `metadata.parquet` live in `{BASE}/images/main/`. To make this shift available to the rest of the team:

1. From a terminal on your local machine, `git pull --rebase` your fork.
2. Copy the updated `images/main/metadata.parquet` into your local clone.
3. `git add images/main/metadata.parquet && git commit -m "shift: <coauthor>-<date> +N imgs" && git push`.

The actual PNG files are *not* committed to git (they would balloon the repo). They live on each coauthor's Drive/Kaggle workspace; the panel builder will fetch them at classification time.

**Cost ledger.** Append a row to `docs/COST_LOG.md`:
```
| YYYY-MM-DD HH:MM UTC | generate shift | Colab T4 free | N imgs / Ts | $0.00 | **$0.00** |
```